In [1]:
# advanced Retrieval-Augmented Generation (RAG), focusing on techniques such as metadata augmentation and metadata filtering using LlamaIndex. This will help you improve your RAG models for better performance and accuracy.


%pip install nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [1]:
import nest_asyncio
nest_asyncio.apply()

In [1]:
%pip install -Uq llama-index

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install -Uq llama-index-embeddings-huggingface

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.24.4 which is incompatible.


In [1]:
#Extract
#To read the designated directory
from llama_index.core import SimpleDirectoryReader

docs = SimpleDirectoryReader(input_dir="./data").load_data()

# file name as id
# docs_nam_as_id = SimpleDirectoryReader(input_dir="./data", filename_as_id=True).load_data()

In [2]:
len(docs)

30

In [3]:
import pprint
pprint.pprint(docs)

[Document(id_='c7b58bd6-75a1-460b-9ffa-c122cbe6eb12', embedding=None, metadata={'page_label': '1', 'file_name': 'energies-18-00391.pdf', 'file_path': 'C:\\Users\\tayos\\PycharmProjects\\JupyterProject\\data\\energies-18-00391.pdf', 'file_type': 'application/pdf', 'file_size': 5316085, 'creation_date': '2025-05-19', 'last_modified_date': '2025-05-19'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='Academic Editor: Reza Rezaee\nReceived: 25 December 2024\nRevised: 12 January 2025\nAccepted: 15 January 2025\nPublished: 17 January 2025\nCitation: Chen, F.; Sun, L.; Jiang, B.;\nHuo, X.; Pan, X.; Feng, C.; Zhang, Z. A\nReview of AI Application

In [4]:
#Transform
# hide some keys from llm

docs[0].__dict__ # too much data about one doc

{'id_': 'c7b58bd6-75a1-460b-9ffa-c122cbe6eb12',
 'embedding': None,
 'metadata': {'page_label': '1',
  'file_name': 'energies-18-00391.pdf',
  'file_path': 'C:\\Users\\tayos\\PycharmProjects\\JupyterProject\\data\\energies-18-00391.pdf',
  'file_type': 'application/pdf',
  'file_size': 5316085,
  'creation_date': '2025-05-19',
  'last_modified_date': '2025-05-19'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {},
 'metadata_template': '{key}: {value}',
 'metadata_separator': '\n',
 'text_resource': MediaResource(embeddings=None, data=None, text='Academic Editor: Reza Rezaee\nReceived: 25 December 2024\nRevised: 12 January 2025\nAccepted: 15 January 2025\nPublished: 17 January 2025\nCitation: Chen, F.; Sun, L.; Jiang, B.;\nHuo, X.

In [5]:
# quick example of what the LLM and Embeddings see when with a test document

from llama_index.core import Document
from llama_index.core.schema import MetadataMode

document = Document(
    text="This is a super-customized document",
    metadata={
        "file_name": "super_secret_document.txt",
        "category": "finance",
        "author": "LlamaIndex",
    },
    # excluded_embed_metadata_keys=["file_name"],
    excluded_llm_metadata_keys=["category"],
    metadata_seperator="\n",
    metadata_template="{key}:{value}",
    text_template="Metadata:\n{metadata_str}\n-----\nContent:\n{content}",
)

print(
    "The LLM sees this: \n",
    document.get_content(metadata_mode=MetadataMode.LLM),
)
# print(
#     "The Embedding model sees this: \n",
#     document.get_content(metadata_mode=MetadataMode.EMBED),
# )

The LLM sees this: 
 Metadata:
file_name:super_secret_document.txt
author:LlamaIndex
-----
Content:
This is a super-customized document


In [6]:
from llama_index.core.schema import MetadataMode

print(docs[0].get_content(metadata_mode=MetadataMode.LLM))   # what the llm sees
#print(docs[0].get_content(metadata_mode=MetadataMode.EMBED)) # what embeddings see. in this case, same thing

page_label: 1
file_path: C:\Users\tayos\PycharmProjects\JupyterProject\data\energies-18-00391.pdf

Academic Editor: Reza Rezaee
Received: 25 December 2024
Revised: 12 January 2025
Accepted: 15 January 2025
Published: 17 January 2025
Citation: Chen, F.; Sun, L.; Jiang, B.;
Huo, X.; Pan, X.; Feng, C.; Zhang, Z. A
Review of AI Applications in
Unconventional Oil and Gas
Exploration and Development.
Energies 2025, 18, 391. https://
doi.org/10.3390/en18020391
Copyright: © 2025 by the authors.
Licensee MDPI, Basel, Switzerland.
This article is an open access article
distributed under the terms and
conditions of the Creative Commons
Attribution (CC BY) license
(https://creativecommons.org/
licenses/by/4.0/).
Review
A Review of AI Applications in Unconventional Oil and Gas
Exploration and Development
Feiyu Chen 1,2
 , Linghui Sun 1,2,3, *, Boyu Jiang 1,2 , Xu Huo 1,2
 , Xiuxiu Pan 1,2 , Chun Feng 3 and Zhirong Zhang 1,2
1 University of Chinese Academy of Sciences, Beijing 100049, China; chenfei

In [7]:
for doc in docs:
    # define the content/metadata template
    doc.text_template = "Metadata:\n{metadata_str}\n---\nContent:\n{content}"

    # exclude page label from embedding
    if "page_label" not in doc.excluded_embed_metadata_keys:
        doc.excluded_embed_metadata_keys.append("page_label")

In [8]:
# after editing the content seen by embedings

print(docs[0].get_content(metadata_mode=MetadataMode.EMBED))

Metadata:
file_path: C:\Users\tayos\PycharmProjects\JupyterProject\data\energies-18-00391.pdf
---
Content:
Academic Editor: Reza Rezaee
Received: 25 December 2024
Revised: 12 January 2025
Accepted: 15 January 2025
Published: 17 January 2025
Citation: Chen, F.; Sun, L.; Jiang, B.;
Huo, X.; Pan, X.; Feng, C.; Zhang, Z. A
Review of AI Applications in
Unconventional Oil and Gas
Exploration and Development.
Energies 2025, 18, 391. https://
doi.org/10.3390/en18020391
Copyright: © 2025 by the authors.
Licensee MDPI, Basel, Switzerland.
This article is an open access article
distributed under the terms and
conditions of the Creative Commons
Attribution (CC BY) license
(https://creativecommons.org/
licenses/by/4.0/).
Review
A Review of AI Applications in Unconventional Oil and Gas
Exploration and Development
Feiyu Chen 1,2
 , Linghui Sun 1,2,3, *, Boyu Jiang 1,2 , Xu Huo 1,2
 , Xiuxiu Pan 1,2 , Chun Feng 3 and Zhirong Zhang 1,2
1 University of Chinese Academy of Sciences, Beijing 100049, China;

In [11]:
#Here are other, more advanced transformations. Some require an LLM to work.
# using openai to extract information like title , set of questions and answer that the particular information from the document can help answer
# install the OpenAI LLM plugin for llama-index (and the openai client)
%pip install -Uq llama-index-llms-openai openai


Note: you may need to restart the kernel to use updated packages.


In [10]:
from llama_index.llms.openai import OpenAI
import os
import getpass

# Prompt for your OpenAI key and set it
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")


In [11]:

# instantiate an OpenAI LLM
llm_transformations = OpenAI(
    model="gpt-3.5-turbo",
    temperature=0.0,
    openai_api_key=os.environ["OPENAI_API_KEY"]
)


In [12]:
from llama_index.llms.openai import OpenAI
from llama_index.core.extractors import (
    TitleExtractor,
    QuestionsAnsweredExtractor,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline
import os
import getpass


# 3. Define your transformations
#
text_splitter = SentenceSplitter(
    separator=" ",
    chunk_size=1024,
    chunk_overlap=128
)
title_extractor = TitleExtractor(
    llm=llm_transformations,
    nodes=5
)
qa_extractor = QuestionsAnsweredExtractor(
    llm=llm_transformations,
    questions=3
)

# 4. Wire them into an ingestion pipeline
pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        title_extractor,
        qa_extractor
    ]
)

# 5. Run the pipeline over your loaded docs
nodes = pipeline.run(
    documents=docs,
    in_place=True,
    show_progress=True,
)


Parsing nodes:   0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 42/42 [00:14<00:00,  2.84it/s]


In [13]:
len(nodes)

42

In [14]:
import pprint

# pprint.pprint(nodes[0].__dict__)

print(nodes[0].get_content(metadata_mode=MetadataMode.LLM))

[Excerpt from document]
page_label: 1
file_path: C:\Users\tayos\PycharmProjects\JupyterProject\data\energies-18-00391.pdf
document_title: The Role of Artificial Intelligence in Unconventional Oil and Gas Exploration and Development: A Comprehensive Review
questions_this_excerpt_can_answer: 1. How has artificial intelligence (AI) been utilized in unconventional oil and gas exploration and development, and what specific applications have shown promising results?
2. What are the current challenges faced in the application of AI in the oil and gas industry, particularly in relation to data quality, model generalization, and interpretability?
3. What future research directions are proposed in the context of AI applications in unconventional oil and gas exploration and development, and how can these advancements improve efficiency and sustainability in the industry?
Excerpt:
-----
Academic Editor: Reza Rezaee
Received: 25 December 2024
Revised: 12 January 2025
Accepted: 15 January 2025
Publi

In [15]:
%pip install -Uq llama-index-embeddings-huggingface

Note: you may need to restart the kernel to use updated packages.


In [16]:
# Embeddings

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

hf_embeddings = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

test_embed = hf_embeddings.get_text_embedding("Hello world")
print(test_embed)

[0.015196128748357296, -0.022570643573999405, 0.008547152392566204, -0.07417063415050507, 0.0038364059291779995, 0.0027134979609400034, -0.031267862766981125, 0.044634006917476654, 0.04405515640974045, -0.00787118636071682, -0.02520081214606762, -0.03336655721068382, 0.014427912421524525, 0.04653817415237427, 0.00855511799454689, -0.01614576391875744, 0.007405844982713461, -0.01901249587535858, -0.114726223051548, -0.01815761625766754, 0.12635937333106995, 0.02970292419195175, 0.025281013920903206, -0.03421786054968834, -0.040999673306941986, 0.00661730719730258, 0.010270638391375542, 0.022362276911735535, 0.004436304792761803, -0.1273096650838852, -0.01614922471344471, -0.020380167290568352, 0.0472121424973011, 0.011579877696931362, 0.0681871846318245, 0.007298651617020369, -0.017853040248155594, 0.04078219085931778, -0.010269420221447945, 0.02375705912709236, 0.010602855123579502, -0.028584381565451622, 0.008159682154655457, -0.015180527232587337, 0.030896255746483803, -0.06597992777

In [17]:
# create index

from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes, embed_model=hf_embeddings)

In [18]:
#query

from llama_index.llms.openai import OpenAI

# instantiate an OpenAI LLM for querying
llm_querying = OpenAI(
    model="gpt-4",
    temperature=0.0,
    openai_api_key=os.environ["OPENAI_API_KEY"]
)

# plug it into your query engine
query_engine = index.as_query_engine(llm=llm_querying)
response = query_engine.query("What are challenges faced in the Oil industry? ")

print(response)


The oil industry, particularly in the development of unconventional oil and gas, faces several challenges. Firstly, there are complex geological conditions. Unconventional reservoirs typically exhibit low porosity, low permeability, nanoporous media, complex lithology, and strong heterogeneity. These characteristics complicate the flow mechanisms of oil and gas, significantly increasing the difficulty of reservoir evaluation and production forecasting. 

Secondly, there are technological and cost challenges. Traditional drilling and completion technologies are inadequate in meeting the demands of unconventional oil and gas development, necessitating the adoption of advanced techniques such as horizontal drilling and multistage hydraulic fracturing. However, these technologies encounter technical bottlenecks and high development costs. The success rate of drilling operations is relatively low, drilling cycles are prolonged, fracturing effectiveness is limited, and it is difficult to imp

In [19]:
# to check the full schema of the response
response.__dict__

{'response': 'The oil industry, particularly in the development of unconventional oil and gas, faces several challenges. Firstly, there are complex geological conditions. Unconventional reservoirs typically exhibit low porosity, low permeability, nanoporous media, complex lithology, and strong heterogeneity. These characteristics complicate the flow mechanisms of oil and gas, significantly increasing the difficulty of reservoir evaluation and production forecasting. \n\nSecondly, there are technological and cost challenges. Traditional drilling and completion technologies are inadequate in meeting the demands of unconventional oil and gas development, necessitating the adoption of advanced techniques such as horizontal drilling and multistage hydraulic fracturing. However, these technologies encounter technical bottlenecks and high development costs. The success rate of drilling operations is relatively low, drilling cycles are prolonged, fracturing effectiveness is limited, and it is 

In [20]:
# persistent storage so you dont have to re embed everything every time. This will keep the index persistent in the machine. This saves time and money by not regenerating the vectors

#STORE
index.storage_context.persist(persist_dir="./vectors")

In [21]:
from llama_index.core import StorageContext, load_index_from_storage

# rebuild storage context
storage_context = StorageContext.from_defaults(persist_dir="./vectors")

# load index
index_from_storage = load_index_from_storage(storage_context, embed_model=hf_embeddings)

In [22]:
qa = index_from_storage.as_query_engine(llm=llm_querying)

In [23]:
response = qa.query("what is reservoir engineering?")
print(response)

Reservoir engineering is a branch of petroleum engineering that applies scientific principles to the fluid flow through porous medium during the development and production of oil and gas reservoirs. It involves the estimation of oil and gas reserves and the prediction of future oil and gas recovery from underground deposits, often through the use of complex mathematical models.


In [30]:
#Using Vector stores
%pip install -Uq chromadb
%pip install -Uq llama-index-vector-stores-chroma

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 5.2.2 requires pyqt5<5.13, which is not installed.
spyder 5.2.2 requires pyqtwebengine<5.13, which is not installed.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
scipy 1.9.1 requires numpy<1.25.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 2.0.2 which is incompatible.
langchain 0.0.352 requires numpy<2,>=1, but you have numpy 2.0.2 which is incompatible.
langchain-community 0.0.6 requires numpy<2,>=1, but you have numpy 2.0.2 which is incompatible.
jupyter-server 1.18.1 requires anyio<4,>=3.1.0, but you have anyio 4.9.0 which is incompatible.
conda-repo-cli 1.0.20 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.20 requires nbformat==5.4.0, but you have nbformat 5.5.0 whi

Note: you may need to restart the kernel to use updated packages.


In [24]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# initialize client, setting path to save data
db = chromadb.PersistentClient(path="./chroma_db")

# create collection
chroma_collection = db.get_or_create_collection("OilGpt")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# create your index
index = VectorStoreIndex(
    nodes, storage_context=storage_context, embed_model=hf_embeddings
)

# You can also load from documents and apply transformations in place
# index = VectorStoreIndex.from_documents(
#     documents, storage_context=storage_context, transformations=[]
# )

# Or you can initialize your index from your vector store and then add the nodes
# index = VectorStoreIndex.from_vector_store(
#     vector_store=vector_store, embed_model=hf_embeddings
# )
# index.insert_nodes(nodes)


# create a query engine and query
query_engine = index.as_query_engine(llm=llm_querying)

In [25]:
response = query_engine.query("What is reservoir engineering?")
print(response)

Reservoir engineering is a branch of petroleum engineering that applies scientific principles to the fluid flow through porous medium during the development and production of oil and gas reservoirs. It involves the estimation of oil and gas reserves and the prediction of future oil and gas recovery from underground deposits. It also includes the design and implementation of efficient methods for the extraction of these resources.
